In [ ]:
import pandas as pd
import numpy as np
import h5py

path = 'data/ca_his_raw_2019.h5'
with h5py.File(path, "r") as f:
    print(list(f.keys()))
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, obj.shape, obj.dtype)

    f.visititems(show)

['t']
t/axis0 (8600,) |S9
t/axis1 (105120,) int64
t/block0_items (8600,) |S9
t/block0_values (105120, 8600) float64


In [147]:
domain = pd.read_csv('data/ca_meta.csv', dtype={"ID": str})
domain.head()

,ID,Lat,Lng,District,County,Fwy,Lanes,Type,Direction,ID2
0,317802,38.389811,-121.479587,3,Sacramento,I5-N,2,Mainline,N,0
1,312134,38.412564,-121.484319,3,Sacramento,I5-N,2,Mainline,N,1
2,312133,38.428630,-121.487657,3,Sacramento,I5-N,3,Mainline,N,2
3,313159,38.450246,-121.492176,3,Sacramento,I5-N,3,Mainline,N,3
4,319767,38.465539,-121.496450,3,Sacramento,I5-N,3,Mainline,N,4


In [152]:
domain_fwy = ['SR22-E','SR22-W','SR55-N','SR55-S']
domain.loc[domain['Fwy'].isin(domain_fwy), 'ID'].to_string(index=False).split('\n')

['1202590',
 '1214853',
 '1215236',
 '1202595',
 '1214869',
 '1202614',
 '1215092',
 '1202627',
 '1202648',
 '1202691',
 '1214938',
 '1214955',
 '1202705',
 '1202720',
 '1202742',
 '1202753',
 '1214988',
 '1215003',
 '1214805',
 '1202779',
 '1202785',
 '1215252',
 '1214894',
 '1202827',
 '1215017',
 '1202844',
 '1202855',
 '1215043',
 '1202885',
 '1214715',
 '1202901',
 '1212170',
 '1202921',
 '1202949',
 '1202964',
 '1202977',
 '1214881',
 '1215026',
 '1202574',
 '1202564',
 '1214854',
 '1215248',
 '1202599',
 '1214871',
 '1202610',
 '1215091',
 '1202631',
 '1202663',
 '1202676',
 '1214939',
 '1214954',
 '1202701',
 '1202724',
 '1202738',
 '1214972',
 '1214987',
 '1215002',
 '1214806',
 '1202766',
 '1202789',
 '1215250',
 '1202803',
 '1202814',
 '1215018',
 '1202840',
 '1202859',
 '1215044',
 '1202872',
 '1211641',
 '1202912',
 '1202929',
 '1202917',
 '1202938',
 '1202960',
 '1202981',
 '1214882',
 '1202993',
 '1203021',
 '1203035',
 '1203057',
 '1203071',
 '1203082',
 '1210205',
 '12

In [153]:
def extract_details(fwy):
    with h5py.File(path, "r") as f:
        id = domain.loc[domain['Fwy'].isin(domain_fwy), 'ID'].to_string(index=False).split('\n')
        sensors_raw = [x.decode("utf-8") for x in f["t/axis0"][:]]
        sensors = []
        for i in sensors_raw:
            if i in id:
                sensors.append(i)
        timestamps = pd.to_datetime(f['t/axis1'][:])

        print('number of sensors:', len(sensors))
        print('first sensors:', sensors[:20])
        print('first timestamps:', timestamps[:5])
        return sensors, len(sensors), timestamps

domain_fwy = ['SR22-E','SR22-W','SR55-N','SR55-S']
domain_sensors, len_of_sensors, timestamps = extract_details(domain_fwy)

number of sensors: 141
first sensors: ['1202590', '1214853', '1215236', '1202595', '1214869', '1202614', '1215092', '1202627', '1202648', '1202691', '1214938', '1214955', '1202705', '1202720', '1202742', '1202753', '1214988', '1215003', '1214805', '1202779']
first timestamps: DatetimeIndex(['2019-01-01 00:00:00', '2019-01-01 00:05:00',
               '2019-01-01 00:10:00', '2019-01-01 00:15:00',
               '2019-01-01 00:20:00'],
              dtype='datetime64[ns]', freq=None)


In [199]:
def get_df(sensors, timestamps, span):
    wanted = pd.Index(sensors)
    with h5py.File(path, "r") as f:
        hdf_sensors = pd.Index(x.decode("utf-8") for x in f["t/axis0"][:])
        positions = hdf_sensors.get_indexer(wanted)

        values = f['t/block0_values'][:, positions]

    df = pd.DataFrame(values, index=timestamps, columns=wanted)
    if span == 1:
        df = df.loc['2019-01-01':'2019-06-30']
    elif span == 2:
        df = df.loc['2019-07-01':'2019-12-31']
    else:
        df = df
    return df.resample('15min').mean()
domain_df = get_df(domain_sensors, timestamps, 3)
domain_df

,1202590,1214853,1215236,1202595,1214869,1202614,1215092,1202627,1202648,1202691,...,1215809,1212935,1203361,1203387,1212840,1212865,1213274,1212903,1212957,1203440
2019-01-01 00:00:00,47.333333,51.000000,50.666667,49.666667,52.333333,13.333333,9.666667,48.333333,51.000000,62.000000,...,125.000000,51.333333,71.666667,63.000000,64.333333,34.333333,270.000000,48.333333,23.666667,30.333333
2019-01-01 00:15:00,78.666667,91.000000,87.666667,87.000000,88.666667,72.000000,58.333333,77.666667,102.666667,131.000000,...,255.666667,126.666667,179.000000,150.333333,145.666667,199.666667,343.333333,147.666667,148.000000,162.000000
2019-01-01 00:30:00,112.333333,130.666667,129.000000,126.666667,127.000000,124.666667,118.000000,116.000000,136.000000,169.000000,...,315.666667,167.333333,226.666667,173.666667,171.333333,295.666667,383.333333,190.333333,184.333333,192.333333
2019-01-01 00:45:00,124.666667,141.333333,138.333333,137.666667,140.000000,134.000000,124.000000,124.666667,140.333333,171.666667,...,331.333333,179.333333,245.000000,214.000000,205.666667,290.000000,416.333333,233.000000,213.666667,226.333333
2019-01-01 01:00:00,115.000000,136.333333,132.333333,130.333333,135.666667,129.666667,120.666667,121.333333,128.000000,152.000000,...,327.000000,177.666667,264.000000,198.333333,195.333333,252.666667,421.000000,238.000000,220.333333,228.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 22:45:00,92.666667,108.333333,106.000000,106.666667,111.000000,103.666667,98.000000,97.666667,104.333333,126.333333,...,160.333333,126.000000,173.333333,154.666667,149.000000,173.000000,141.000000,147.666667,131.000000,128.333333
2019-12-31 23:00:00,78.333333,97.666667,93.333333,92.000000,96.333333,90.333333,86.000000,85.333333,93.000000,111.666667,...,116.333333,92.666667,130.333333,112.333333,107.333333,123.666667,98.666667,111.666667,102.000000,96.000000
2019-12-31 23:15:00,75.666667,89.666667,90.333333,90.333333,91.333333,91.000000,84.666667,85.333333,87.666667,106.000000,...,113.000000,95.000000,151.000000,128.666667,130.333333,147.333333,115.666667,128.333333,135.000000,113.666667
2019-12-31 23:30:00,68.333333,78.000000,75.333333,75.666667,75.666667,71.666667,69.000000,70.000000,78.000000,95.000000,...,108.000000,85.666667,116.333333,104.666667,107.666667,120.000000,99.333333,110.000000,104.000000,93.666667


In [200]:
def sensor_to_fwy(df):
    return domain.set_index("ID").loc[df.columns, "Fwy"]
domain_sensor_to_fwy = sensor_to_fwy(domain_df)
domain_sensor_to_fwy

1202590    SR22-E
1214853    SR22-E
1215236    SR22-E
1202595    SR22-E
1214869    SR22-E
            ...  
1212865    SR55-S
1213274    SR55-S
1212903    SR55-S
1212957    SR55-S
1203440    SR55-S
Name: Fwy, Length: 141, dtype: object

In [201]:
def mean_flow(df, fwy):
    return df.T.groupby(fwy).apply(lambda x: np.nanmean(x.to_numpy()))

mean_flow_by_freeway = mean_flow(domain_df, domain_sensor_to_fwy)
mean_flow_by_freeway

Fwy
SR22-E    269.869467
SR22-W    292.518164
SR55-N    313.565661
SR55-S    318.288118
dtype: float64

In [92]:
profile = pd.DataFrame(columns = df.columns)
for i in range(0,24):
    profile.loc[i] = df.loc[df.index.hour == i].mean()
profile

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
0,54.402784,58.899135,59.833922,54.752411,57.467690,51.155307,57.891727,57.733497,54.990449,58.344353,...,41.923384,60.829885,53.550972,54.872552,57.504110,54.931699,60.079453,57.303016,59.887247,55.123953
1,56.000108,60.462182,61.311919,56.809890,60.371513,52.562580,60.098961,59.327418,54.794354,58.234049,...,41.125595,62.459854,53.198970,54.837849,57.848468,55.908984,60.814164,58.362324,60.370896,56.703431
2,54.482799,59.588340,60.535748,55.117260,60.112668,51.739179,59.595692,58.852207,52.671596,56.423047,...,38.330626,60.804146,51.696933,53.439869,55.064978,54.913042,59.865331,57.523107,58.534876,55.504410
3,53.302668,58.405776,60.480721,53.016159,59.137356,51.255506,58.577716,58.153440,52.457773,56.779119,...,35.094428,59.099767,47.903032,52.743001,53.300734,53.179901,58.388167,57.241364,56.449158,55.050868
4,54.511905,57.384806,61.870995,53.365304,59.933119,52.673467,59.193687,56.978945,55.131709,59.810267,...,36.251514,59.627042,48.267414,54.967271,55.192250,53.763057,59.444696,59.408588,57.741506,57.558812
5,56.323488,57.445599,62.246757,53.489054,60.098298,55.125199,59.730109,57.523872,55.309624,59.455758,...,40.573959,61.139022,51.193527,57.182569,57.566785,54.414332,60.243193,59.709971,58.382920,58.844359
6,57.395041,58.197271,59.830624,50.685146,54.163532,53.581618,60.108847,57.595685,44.061698,36.275579,...,42.005005,61.341382,53.069283,59.222946,58.858313,55.784853,61.204961,52.078024,59.683227,54.148163
7,58.580837,59.650135,41.709217,48.404062,45.164337,49.959584,59.930418,57.658901,46.775452,25.666404,...,38.732268,61.981313,52.837234,60.918731,59.276269,57.570999,62.455120,46.544910,60.288515,53.309424
8,57.200023,59.478033,38.485557,48.227154,41.614477,50.253208,58.941970,57.502238,41.616664,25.185860,...,36.256943,61.989212,52.735119,60.641499,47.760938,57.603489,61.661129,44.268744,59.464175,45.913410
9,56.459344,59.246459,54.235045,46.288348,39.843467,48.693591,57.214124,57.676992,46.276999,26.467733,...,39.387603,60.990157,52.054134,59.935765,52.491875,55.780960,60.804158,45.589590,59.695704,48.696430


In [100]:
from datetime import datetime, timedelta

def recent_twelve_readings(time, min = 15, freq = 12):
    span = timedelta(minutes = min * freq-1)
    window = df.loc[time - span : time]
    return window

window = recent_twelve_readings(pd.Timestamp("2012-03-10 04:15:00"))
window

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-10 01:30:00,63.375000,66.750000,63.625000,60.625000,68.875000,67.125000,67.125000,65.750000,59.000000,60.375000,...,43.500000,69.000000,0.0,48.375000,61.375000,63.625000,68.625000,61.625000,67.000000,62.125000
2012-03-10 01:45:00,67.625000,66.750000,67.625000,60.750000,64.500000,56.571429,66.250000,64.375000,65.000000,67.750000,...,45.000000,68.750000,0.0,61.000000,69.500000,64.625000,64.250000,62.125000,68.750000,61.125000
2012-03-10 02:00:00,66.000000,66.500000,67.750000,60.000000,66.250000,64.571429,65.625000,66.625000,61.750000,62.250000,...,41.250000,68.571429,0.0,57.625000,69.875000,66.375000,67.750000,60.125000,68.625000,63.285714
2012-03-10 02:15:00,65.000000,65.875000,64.625000,63.500000,66.375000,68.875000,64.875000,65.875000,53.250000,69.428571,...,47.625000,68.500000,0.0,61.250000,62.875000,62.125000,67.500000,62.250000,69.500000,63.125000
2012-03-10 02:30:00,64.222222,67.222222,68.444444,60.333333,67.333333,66.555556,65.444444,66.111111,61.777778,61.666667,...,44.333333,65.666667,0.0,66.111111,55.222222,62.888889,66.444444,63.333333,67.444444,60.888889
2012-03-10 02:45:00,59.666667,66.888889,68.000000,63.111111,67.444444,67.111111,67.555556,66.333333,57.000000,66.444444,...,44.222222,61.333333,0.0,50.250000,62.111111,59.444444,65.888889,60.222222,67.000000,62.333333
2012-03-10 03:00:00,65.500000,65.750000,69.500000,60.875000,68.625000,67.500000,62.875000,61.375000,63.125000,67.375000,...,46.714286,65.250000,0.0,56.750000,63.125000,67.625000,68.125000,65.142857,69.750000,62.875000
2012-03-10 03:15:00,62.875000,62.000000,67.375000,62.625000,68.375000,66.000000,67.000000,63.250000,62.000000,66.857143,...,40.000000,67.000000,0.0,54.375000,66.500000,61.000000,65.375000,68.125000,65.250000,61.250000
2012-03-10 03:30:00,60.375000,65.125000,64.750000,57.000000,66.625000,64.125000,66.375000,67.375000,62.750000,65.500000,...,45.500000,66.625000,0.0,57.750000,63.750000,65.000000,68.875000,65.250000,65.750000,58.375000
2012-03-10 03:45:00,53.250000,68.000000,68.375000,61.125000,64.625000,66.125000,64.750000,63.750000,64.500000,67.500000,...,40.875000,65.125000,0.0,59.000000,66.625000,61.250000,68.000000,64.250000,68.500000,61.750000


In [94]:
row, col = window.shape
row

2

Timestamp('2012-03-10 04:15:00')

In [132]:
from datetime import datetime, timedelta

def short_term_trend(readings, freq = 4):
    subwindow = readings.iloc[-freq:]
    trends = pd.DataFrame(columns= df.columns)
    old_value = subwindow.iloc[0]
    new_value = subwindow.iloc[-1]
    trends = (new_value - old_value)/ (freq-1)
    return subwindow, trends

subwindow, trends = short_term_trend(window)

In [133]:
subwindow

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-10 03:30:00,60.375,65.125,64.750,57.000,66.625,64.125,66.375,67.375,62.75,65.500,...,45.500,66.625,0.0,57.750,63.750,65.000,68.875,65.25,65.75,58.375
2012-03-10 03:45:00,53.250,68.000,68.375,61.125,64.625,66.125,64.750,63.750,64.50,67.500,...,40.875,65.125,0.0,59.000,66.625,61.250,68.000,64.25,68.50,61.750
2012-03-10 04:00:00,62.200,66.200,62.600,61.400,67.800,59.400,63.600,62.400,47.80,68.800,...,39.800,59.200,0.0,60.400,57.000,69.200,68.400,66.40,68.20,58.800
2012-03-10 04:15:00,60.000,66.125,68.000,56.875,63.750,62.000,62.625,68.125,58.25,66.375,...,40.125,64.375,0.0,59.375,60.375,58.625,65.125,65.00,65.00,63.125


In [134]:
trends

773869   -0.125000
767541    0.333333
767542    1.083333
717447   -0.041667
717446   -0.958333
            ...   
717592   -2.125000
717595   -1.250000
772168   -0.083333
718141   -0.250000
769373    1.583333
Length: 207, dtype: float64